# Efficient Attention and Inference — Exercises

8 exercises covering roofline analysis, KV cache sizing, speculative decoding,
FlashAttention IO, quantisation, PagedAttention, continuous batching, and
prefill-decode disaggregation.

**Exercises:**
1. Roofline Analysis — ridge point and decode regime
2. KV Cache Sizing — memory per request and concurrent capacity
3. Speculative Decoding Speedup — expected tokens and speedup
4. FlashAttention IO — standard vs FlashAttention HBM access
5. Quantisation Tradeoff — model size and decode speed
6. PagedAttention Fragmentation — static vs paged memory waste
7. Continuous Batching Throughput — waste reduction
8. Prefill-Decode Disaggregation — optimal GPU ratio

Each exercise has: **description → scaffold → solution**

[← Theory Notebook](theory.ipynb) | [Notes](notes.md)

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def fmt_bytes(b):
    """Format byte count."""
    if b >= 1e12:
        return f'{b/1e12:.2f} TB'
    elif b >= 1e9:
        return f'{b/1e9:.2f} GB'
    elif b >= 1e6:
        return f'{b/1e6:.1f} MB'
    elif b >= 1e3:
        return f'{b/1e3:.1f} KB'
    return f'{b:.0f} B'

def print_table(headers, rows, col_widths=None):
    """Pretty-print a table."""
    if col_widths is None:
        col_widths = [max(len(str(h)), max(len(str(r[i])) for r in rows))
                      for i, h in enumerate(headers)]
    header_str = ' | '.join(str(h).ljust(w) for h, w in zip(headers, col_widths))
    print(header_str)
    print('-' * len(header_str))
    for row in rows:
        print(' | '.join(str(v).ljust(w) for v, w in zip(row, col_widths)))

def roofline_throughput(intensity, peak_flops, peak_bw):
    """Compute achievable throughput from roofline model."""
    return min(peak_flops, peak_bw * intensity)

print("Setup complete ✓")

---
## Exercise 1: Roofline Analysis

**A100 GPU specs:**
- Peak compute: 312 TFLOPS (BF16)
- HBM bandwidth: 2 TB/s

**Tasks:**
1. Compute the ridge point I* (FLOPs/byte) — the boundary between bandwidth-bound
   and compute-bound regimes.
2. For a decode step at batch=1, s=4096, d_model=4096: compute the arithmetic
   intensity of a linear layer Y = x @ W where x is [1, 4096] and W is [4096, 4096].
3. Is this decode step compute-bound or bandwidth-bound?

In [ ]:
# === Exercise 1: Scaffold ===

# A100 specs
peak_flops = 312e12       # 312 TFLOPS
peak_bandwidth = 2.0e12   # 2 TB/s

# TODO 1: Compute ridge point I* = peak_flops / peak_bandwidth
ridge_point = None  # fill in

# Decode step: linear layer Y = x @ W
batch = 1
d_model = 4096
bytes_per_elem = 2  # BF16

# TODO 2: Compute FLOPs for Y = x @ W
#   FLOPs = 2 * batch * d_model * d_model  (multiply-accumulate)
flops = None  # fill in

# TODO 3: Compute bytes accessed
#   Bytes = W (d*d*2) + x (batch*d*2) + Y (batch*d*2)
bytes_accessed = None  # fill in

# TODO 4: Compute arithmetic intensity
intensity = None  # fill in

# TODO 5: Determine regime — compare intensity to ridge_point
regime = None  # 'bandwidth-bound' or 'compute-bound'

In [ ]:
# === Exercise 1: Solution ===
np.random.seed(42)

print("=== Exercise 1: Roofline Analysis ===\n")

# Step 1: Ridge point
peak_flops = 312e12
peak_bandwidth = 2.0e12
ridge_point = peak_flops / peak_bandwidth
print(f"Ridge point I* = {peak_flops/1e12:.0f} TFLOPS / {peak_bandwidth/1e12:.1f} TB/s")
print(f"              = {ridge_point:.0f} FLOPs/byte")

# Step 2: FLOPs for linear layer
batch = 1
d_model = 4096
bytes_per_elem = 2

flops = 2 * batch * d_model * d_model
print(f"\nFLOPs = 2 × {batch} × {d_model} × {d_model} = {flops:,}")

# Step 3: Bytes accessed
w_bytes = d_model * d_model * bytes_per_elem
x_bytes = batch * d_model * bytes_per_elem
y_bytes = batch * d_model * bytes_per_elem
bytes_accessed = w_bytes + x_bytes + y_bytes
print(f"Bytes = W({fmt_bytes(w_bytes)}) + x({fmt_bytes(x_bytes)}) + Y({fmt_bytes(y_bytes)})")
print(f"      = {fmt_bytes(bytes_accessed)}")

# Step 4: Arithmetic intensity
intensity = flops / bytes_accessed
print(f"\nArithmetic intensity I = {flops:,} / {bytes_accessed:,} = {intensity:.2f} FLOPs/byte")

# Step 5: Regime
regime = 'compute-bound' if intensity >= ridge_point else 'bandwidth-bound'
print(f"\nI = {intensity:.2f}  vs  I* = {ridge_point:.0f}")
print(f"Regime: {regime.upper()}")
print(f"\nAt batch=1, decode is massively bandwidth-bound ({intensity:.2f} << {ridge_point:.0f}).")
print(f"The GPU compute units are ~{ridge_point/intensity:.0f}× underutilised.")

# Bonus: what batch size crosses the ridge point?
# I ≈ batch (when d >> batch), so we need batch ≈ I*
print(f"\nBatch needed to reach compute-bound: ~{int(ridge_point)} (solve I(B) = I*)")

---
## Exercise 2: KV Cache Sizing

**LLaMA-3 8B configuration:**
- Layers L = 32
- KV heads H_kv = 8 (GQA)
- Head dimension d_k = 128

**Tasks:**
1. Compute KV cache size for one request at s = 8192 in BF16 (2 bytes/elem).
2. Compute KV cache size for the same request in INT8 (1 byte/elem).
3. GPU has 80 GB total, 16 GB reserved for model weights. How many concurrent
   requests fit in the remaining 64 GB for each precision?

In [ ]:
# === Exercise 2: Scaffold ===

# LLaMA-3 8B config
L = 32
H_kv = 8
d_k = 128
s = 8192

# KV cache formula: 2 * L * H_kv * d_k * s * bytes_per_elem
#   factor 2 = one K + one V per layer

# TODO 1: KV cache in BF16
kv_bf16 = None  # fill in: bytes

# TODO 2: KV cache in INT8
kv_int8 = None  # fill in: bytes

# TODO 3: Concurrent requests
available_mem = 80e9 - 16e9  # 64 GB
n_requests_bf16 = None  # fill in: int(available_mem / kv_bf16)
n_requests_int8 = None  # fill in: int(available_mem / kv_int8)

In [ ]:
# === Exercise 2: Solution ===
np.random.seed(42)

print("=== Exercise 2: KV Cache Sizing ===\n")

L = 32
H_kv = 8
d_k = 128
s = 8192

# Step 1: BF16
kv_bf16 = 2 * L * H_kv * d_k * s * 2  # 2 bytes per BF16
print(f"KV cache formula: 2 × L × H_kv × d_k × s × bytes_per_elem")
print(f"BF16: 2 × {L} × {H_kv} × {d_k} × {s} × 2 = {kv_bf16:,} bytes = {fmt_bytes(kv_bf16)}")

# Step 2: INT8
kv_int8 = 2 * L * H_kv * d_k * s * 1  # 1 byte per INT8
print(f"INT8: 2 × {L} × {H_kv} × {d_k} × {s} × 1 = {kv_int8:,} bytes = {fmt_bytes(kv_int8)}")

# Step 3: Concurrent requests
available_mem = 80e9 - 16e9
n_bf16 = int(available_mem / kv_bf16)
n_int8 = int(available_mem / kv_int8)

print(f"\nAvailable memory: {fmt_bytes(available_mem)}")
print(f"\nConcurrent requests at s={s}:")
print(f"  BF16: {n_bf16} requests  (KV/req = {fmt_bytes(kv_bf16)})")
print(f"  INT8: {n_int8} requests  (KV/req = {fmt_bytes(kv_int8)})")
print(f"  INT8 gives {n_int8 / n_bf16:.0f}× more concurrent requests")

# Bonus: GQA vs MHA
kv_mha = 2 * L * 32 * d_k * s * 2  # MHA: H_kv = H = 32
print(f"\nBonus — if MHA (H_kv=32 instead of 8):")
print(f"  KV cache = {fmt_bytes(kv_mha)} ({kv_mha/kv_bf16:.0f}× larger)")
print(f"  Only {int(available_mem / kv_mha)} concurrent requests")
print(f"  GQA saves {(1 - H_kv/32)*100:.0f}% of KV cache memory")

---
## Exercise 3: Speculative Decoding Speedup

**Setup:**
- Draft model acceptance rate α = 0.8
- K = 4 draft tokens per speculation round
- Draft model cost = 10% of target model cost (draft_cost_ratio = 0.1)

**Tasks:**
1. Compute the expected number of tokens accepted per target call using the
   formula: $E[\text{tokens}] = \frac{1 - \alpha^{K+1}}{1 - \alpha}$
2. Compute the cost of one speculation round (K draft passes + 1 target pass).
3. Compute the speedup vs standard autoregressive decoding.
4. At what α does speculative decoding break even (speedup = 1)?

In [ ]:
# === Exercise 3: Scaffold ===

alpha = 0.8
K = 4
draft_cost_ratio = 0.1  # draft is 10% of target cost

# TODO 1: Expected tokens per target call
#   E = (1 - alpha^(K+1)) / (1 - alpha)
expected_tokens = None  # fill in

# TODO 2: Cost of one speculation round
#   Cost = K * draft_cost + 1 * target_cost
#   Normalise: target_cost = 1.0
spec_cost = None  # fill in

# TODO 3: Speedup
#   Standard cost for expected_tokens = expected_tokens * target_cost
#   Speedup = standard_cost / spec_cost
speedup = None  # fill in

# TODO 4: Break-even alpha
#   Solve: E(alpha, K) / (K * r + 1) = 1
#   i.e. (1 - alpha^(K+1)) / (1 - alpha) = K * r + 1
breakeven_alpha = None  # find numerically

In [ ]:
# === Exercise 3: Solution ===
np.random.seed(42)

print("=== Exercise 3: Speculative Decoding Speedup ===\n")

alpha = 0.8
K = 4
draft_cost_ratio = 0.1

# Step 1: Expected tokens
expected_tokens = (1 - alpha**(K+1)) / (1 - alpha)
print(f"E[tokens] = (1 - {alpha}^{K+1}) / (1 - {alpha})")
print(f"          = (1 - {alpha**(K+1):.5f}) / {1-alpha:.1f}")
print(f"          = {expected_tokens:.4f}")

# Step 2: Cost of speculation round
spec_cost = K * draft_cost_ratio + 1  # normalised to target_cost = 1
print(f"\nSpec cost = {K} × {draft_cost_ratio} + 1 = {spec_cost:.1f} (target units)")

# Step 3: Speedup
standard_cost = expected_tokens * 1.0  # each token costs 1 target call
speedup = standard_cost / spec_cost
print(f"Standard cost = {expected_tokens:.4f} target calls")
print(f"Speedup = {expected_tokens:.4f} / {spec_cost:.1f} = {speedup:.2f}×")

# Step 4: Break-even alpha (numerical search)
print(f"\n--- Break-even Analysis ---")
alphas = np.linspace(0.01, 0.99, 10000)
for a in alphas:
    E = (1 - a**(K+1)) / (1 - a)
    s = E / spec_cost
    if s >= 1.0:
        breakeven_alpha = a
        break

print(f"Break-even α ≈ {breakeven_alpha:.3f}")
print(f"  At α = {breakeven_alpha:.3f}: E = {(1-breakeven_alpha**(K+1))/(1-breakeven_alpha):.3f}, "
      f"speedup = {(1-breakeven_alpha**(K+1))/(1-breakeven_alpha)/spec_cost:.3f}×")
print(f"  Below this α, speculative decoding is slower than standard decoding.")

# Bonus: sweep
print(f"\n--- Speedup Table ---")
rows = []
for a in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    E = (1 - a**(K+1)) / (1 - a)
    s = E / spec_cost
    rows.append([f"{a:.2f}", f"{E:.3f}", f"{s:.2f}×"])
print_table(['α', 'E[tokens]', 'Speedup'], rows)

---
## Exercise 4: FlashAttention IO

**Parameters:**
- Sequence length n = 4096
- Head dimension d = 64
- Number of heads H = 32
- BF16 (2 bytes per element)

**Standard attention** materialises the full n×n attention matrix to HBM:
- Read Q, K → compute S=QK^T → write S to HBM
- Read S → softmax → write P to HBM
- Read P, V → compute O=PV → write O to HBM

**FlashAttention** never materialises the n×n matrix:
- Read Q, K, V once; write O once. Total IO ≈ 4nd bytes.

**Tasks:**
1. Compute standard attention HBM IO per head.
2. Compute FlashAttention HBM IO per head.
3. Compute total IO for all 32 heads (both methods).
4. Compute speedup assuming bandwidth-bound at 2 TB/s.

In [ ]:
# === Exercise 4: Scaffold ===

n = 4096
d = 64
H = 32
bpe = 2  # BF16

# TODO 1: Standard attention IO per head
#   Reads: Q(n*d) + K(n*d) + S(n*n) + P(n*n) + V(n*d)  = 3*n*d + 2*n*n
#   Writes: S(n*n) + P(n*n) + O(n*d)                    = n*d + 2*n*n
#   Total elements: 4*n*d + 4*n*n
#   Total bytes: (4*n*d + 4*n*n) * bpe
io_std_per_head = None  # fill in

# TODO 2: FlashAttention IO per head
#   Reads: Q(n*d) + K(n*d) + V(n*d)  = 3*n*d
#   Writes: O(n*d)                    = n*d
#   Total: 4*n*d * bpe
io_fa_per_head = None  # fill in

# TODO 3: Total IO for H heads
io_std_total = None  # fill in
io_fa_total = None   # fill in

# TODO 4: Speedup at 2 TB/s
bandwidth = 2e12
time_std = None  # io_std_total / bandwidth
time_fa = None   # io_fa_total / bandwidth
speedup = None   # time_std / time_fa

In [ ]:
# === Exercise 4: Solution ===
np.random.seed(42)

print("=== Exercise 4: FlashAttention IO ===\n")

n = 4096
d = 64
H = 32
bpe = 2

# Step 1: Standard attention IO per head
std_elements = 4 * n * d + 4 * n * n   # reads + writes
io_std_per_head = std_elements * bpe
print("Standard attention IO per head:")
print(f"  Reads:  Q({n}×{d}) + K({n}×{d}) + S({n}×{n}) + P({n}×{n}) + V({n}×{d})")
print(f"  Writes: S({n}×{n}) + P({n}×{n}) + O({n}×{d})")
print(f"  Total elements: 4×{n}×{d} + 4×{n}×{n} = {4*n*d:,} + {4*n*n:,} = {std_elements:,}")
print(f"  Total bytes: {std_elements:,} × {bpe} = {fmt_bytes(io_std_per_head)}")

# Step 2: FlashAttention IO per head
fa_elements = 4 * n * d
io_fa_per_head = fa_elements * bpe
print(f"\nFlashAttention IO per head:")
print(f"  Reads:  Q({n}×{d}) + K({n}×{d}) + V({n}×{d})")
print(f"  Writes: O({n}×{d})")
print(f"  Total: 4×{n}×{d} = {fa_elements:,} elements = {fmt_bytes(io_fa_per_head)}")

# Step 3: Total for all heads
io_std_total = io_std_per_head * H
io_fa_total = io_fa_per_head * H
print(f"\nTotal IO for {H} heads:")
print(f"  Standard:      {fmt_bytes(io_std_total)}")
print(f"  FlashAttention: {fmt_bytes(io_fa_total)}")

# Step 4: Speedup
bandwidth = 2e12
time_std = io_std_total / bandwidth * 1000  # ms
time_fa = io_fa_total / bandwidth * 1000
speedup = io_std_total / io_fa_total

print(f"\nAt {bandwidth/1e12:.0f} TB/s bandwidth:")
print(f"  Standard time:      {time_std:.3f} ms")
print(f"  FlashAttention time: {time_fa:.3f} ms")
print(f"  IO speedup: {speedup:.1f}×")
print(f"\nSpeedup = (4nd + 4n²) / (4nd) = 1 + n/d = 1 + {n}/{d} = {1+n/d:.0f}×")
print(f"The longer the sequence relative to head dim, the bigger the speedup.")

---
## Exercise 5: Quantisation Tradeoff

**LLaMA-3 70B** with 70 billion parameters.

**Tasks:**
1. Compute model size in BF16, INT8, and INT4.
2. Compute the maximum decode tokens/sec on H100 (bandwidth = 3.35 TB/s)
   for each format. Assume decode is purely bandwidth-bound at batch=1.
   Formula: max tok/s = bandwidth / model_bytes.
3. Compute how many H100 GPUs (80 GB each) are needed to hold the model
   for each format.

In [ ]:
# === Exercise 5: Scaffold ===

n_params = 70e9
bandwidth_h100 = 3.35e12  # 3.35 TB/s
gpu_mem = 80e9             # 80 GB per GPU

# TODO 1: Model sizes
size_bf16 = None  # n_params * 2 bytes
size_int8 = None  # n_params * 1 byte
size_int4 = None  # n_params * 0.5 bytes

# TODO 2: Max decode tok/s (batch=1, bandwidth-bound)
#   tok/s = bandwidth / model_bytes
toks_bf16 = None  # fill in
toks_int8 = None  # fill in
toks_int4 = None  # fill in

# TODO 3: GPUs needed (ceil(model_size / gpu_mem))
gpus_bf16 = None  # fill in
gpus_int8 = None  # fill in
gpus_int4 = None  # fill in

In [ ]:
# === Exercise 5: Solution ===
np.random.seed(42)

print("=== Exercise 5: Quantisation Tradeoff ===\n")

n_params = 70e9
bandwidth_h100 = 3.35e12
gpu_mem = 80e9

# Step 1: Model sizes
formats = [
    ('BF16', 2),
    ('INT8', 1),
    ('INT4', 0.5),
]

rows = []
for label, bpp in formats:
    size = n_params * bpp
    toks = bandwidth_h100 / size
    gpus = int(np.ceil(size / gpu_mem))
    rows.append([label, f"{bpp}", fmt_bytes(size), f"{toks:.0f}", f"{gpus}"])

print_table(['Format', 'B/param', 'Size', 'Max tok/s', 'GPUs'], rows)

# Analysis
size_bf16 = n_params * 2
size_int4 = n_params * 0.5
toks_bf16 = bandwidth_h100 / size_bf16
toks_int4 = bandwidth_h100 / size_int4

print(f"\nAnalysis:")
print(f"  INT4 is {size_bf16/size_int4:.0f}× smaller than BF16")
print(f"  INT4 decode is {toks_int4/toks_bf16:.0f}× faster (bandwidth-bound)")
print(f"  BF16 needs {int(np.ceil(size_bf16/gpu_mem))} GPUs; INT4 fits on {int(np.ceil(size_int4/gpu_mem))} GPU")
print(f"\n  Tradeoff: quantisation introduces small accuracy loss")
print(f"  but gives {toks_int4/toks_bf16:.0f}× speed + {int(np.ceil(size_bf16/gpu_mem))//int(np.ceil(size_int4/gpu_mem))}× fewer GPUs.")

---
## Exercise 6: PagedAttention Fragmentation

**Setup:**
- 8 concurrent requests
- max_length = 2048 (pre-allocated for static)
- Actual lengths: [128, 256, 512, 1024, 64, 768, 384, 192]
- KV bytes per token = 131,072 (LLaMA-3 8B, BF16: 2×32×8×128×2)

**Tasks:**
1. Compute total memory allocated and wasted with static allocation
   (each request gets max_length slots pre-allocated).
2. Compute total memory allocated and wasted with PagedAttention (block_size=16).
3. Compute memory utilisation for both methods.
4. How many additional requests could fit in the recovered memory?

In [ ]:
# === Exercise 6: Scaffold ===

n_requests = 8
max_length = 2048
actual_lengths = [128, 256, 512, 1024, 64, 768, 384, 192]
kv_per_token = 2 * 32 * 8 * 128 * 2  # 131,072 bytes
block_size = 16

# TODO 1: Static allocation
static_allocated = None  # n_requests * max_length * kv_per_token
static_used = None       # sum(actual_lengths) * kv_per_token
static_waste = None      # allocated - used

# TODO 2: PagedAttention
#   Each request uses ceil(actual_length / block_size) blocks
#   Allocated = total_blocks * block_size * kv_per_token
total_blocks_paged = None  # sum of ceil(l / block_size) for each request
paged_allocated = None
paged_waste = None

# TODO 3: Utilisation
util_static = None  # static_used / static_allocated
util_paged = None   # static_used / paged_allocated

# TODO 4: Additional requests from recovered memory
recovered = None           # static_waste - paged_waste
avg_len = None             # mean of actual_lengths
extra_requests = None      # int(recovered / (avg_len * kv_per_token))

In [ ]:
# === Exercise 6: Solution ===
np.random.seed(42)

print("=== Exercise 6: PagedAttention Fragmentation ===\n")

n_requests = 8
max_length = 2048
actual_lengths = np.array([128, 256, 512, 1024, 64, 768, 384, 192])
kv_per_token = 2 * 32 * 8 * 128 * 2  # 131,072 bytes
block_size = 16

# Step 1: Static allocation
static_allocated = n_requests * max_length * kv_per_token
static_used = np.sum(actual_lengths) * kv_per_token
static_waste = static_allocated - static_used

print(f"Static Allocation:")
print(f"  Allocated: {n_requests} × {max_length} × {kv_per_token:,} = {fmt_bytes(static_allocated)}")
print(f"  Used:      {np.sum(actual_lengths)} × {kv_per_token:,} = {fmt_bytes(static_used)}")
print(f"  Wasted:    {fmt_bytes(static_waste)}")

# Step 2: PagedAttention
blocks_per_req = [int(np.ceil(l / block_size)) for l in actual_lengths]
total_blocks = sum(blocks_per_req)
paged_allocated = total_blocks * block_size * kv_per_token
paged_waste = paged_allocated - static_used

print(f"\nPagedAttention (block_size={block_size}):")
for i, (l, b) in enumerate(zip(actual_lengths, blocks_per_req)):
    internal_frag = b * block_size - l
    print(f"  Req {i}: len={l:>5}, blocks={b:>3}, waste={internal_frag:>3} tokens")
print(f"  Total blocks: {total_blocks}")
print(f"  Allocated: {total_blocks} × {block_size} × {kv_per_token:,} = {fmt_bytes(paged_allocated)}")
print(f"  Wasted:    {fmt_bytes(paged_waste)} (internal fragmentation only)")

# Step 3: Utilisation
util_static = static_used / static_allocated
util_paged = static_used / paged_allocated

print(f"\nUtilisation:")
print(f"  Static:       {util_static:.1%}")
print(f"  PagedAttention: {util_paged:.1%}")
print(f"  Improvement: {util_paged/util_static:.2f}× better utilisation")

# Step 4: Additional requests
recovered = static_waste - paged_waste
avg_len = np.mean(actual_lengths)
avg_kv = int(np.ceil(avg_len / block_size)) * block_size * kv_per_token
extra_requests = int(recovered / avg_kv)

print(f"\nRecovered memory: {fmt_bytes(recovered)}")
print(f"Avg request length: {avg_len:.0f} tokens")
print(f"Additional requests possible: {extra_requests}")
print(f"Total capacity: {n_requests} → {n_requests + extra_requests} ({(n_requests+extra_requests)/n_requests:.1f}× more)")

---
## Exercise 7: Continuous Batching Throughput

**Static batching:**
- Batch size = 8
- Mean output length = 256 tokens
- Max output length = 1024 tokens
- GPU throughput = 2000 tokens/sec (total across batch)

With static batching, the batch waits for the longest request to complete.
Slots for shorter requests sit idle after they finish.

**Tasks:**
1. Estimate the compute waste with static batching (fraction of slot-steps
   that are idle because requests finished early).
2. Compute effective throughput (useful tokens/sec) under static batching.
3. With continuous batching (Orca), estimate improvement in throughput
   assuming new requests fill freed slots immediately.
4. What is the wall-clock time to serve 1000 requests under each method?

In [ ]:
# === Exercise 7: Scaffold ===

batch_size = 8
mean_output = 256
max_output = 1024
gpu_throughput = 2000  # tokens/sec total

# TODO 1: Compute waste with static batching
#   In a static batch, we run max_output steps.
#   Useful tokens = batch_size * mean_output
#   Total slot-steps = batch_size * max_output
#   Waste = 1 - useful/total
useful_tokens = None      # batch_size * mean_output
total_slot_steps = None   # batch_size * max_output
waste_fraction = None     # 1 - useful/total

# TODO 2: Effective throughput
#   Time to complete one batch = max_output steps
#   Tokens produced per batch = batch_size * mean_output
#   Effective throughput = gpu_throughput * (1 - waste)
effective_throughput_static = None  # fill in

# TODO 3: Continuous batching throughput
#   Freed slots immediately filled → no waste → full utilisation
effective_throughput_continuous = None  # gpu_throughput
improvement = None

# TODO 4: Wall-clock time for 1000 requests
n_total = 1000
time_static = None       # fill in
time_continuous = None   # fill in

In [ ]:
# === Exercise 7: Solution ===
np.random.seed(42)

print("=== Exercise 7: Continuous Batching Throughput ===\n")

batch_size = 8
mean_output = 256
max_output = 1024
gpu_throughput = 2000  # tokens/sec

# Step 1: Waste
useful_tokens = batch_size * mean_output
total_slot_steps = batch_size * max_output
waste_fraction = 1 - useful_tokens / total_slot_steps

print(f"Static batching waste:")
print(f"  Useful tokens per batch = {batch_size} × {mean_output} = {useful_tokens}")
print(f"  Total slot-steps = {batch_size} × {max_output} = {total_slot_steps}")
print(f"  Waste = 1 - {useful_tokens}/{total_slot_steps} = {waste_fraction:.1%}")
print(f"  → {waste_fraction:.0%} of GPU compute wasted on padding/idle slots")

# Step 2: Effective throughput
effective_static = gpu_throughput * (1 - waste_fraction)
print(f"\nEffective throughput:")
print(f"  Static: {gpu_throughput} × (1 - {waste_fraction:.2f}) = {effective_static:.0f} useful tok/s")

# Step 3: Continuous batching
effective_continuous = gpu_throughput  # 100% utilisation
improvement = effective_continuous / effective_static
print(f"  Continuous: {effective_continuous:.0f} useful tok/s (no waste)")
print(f"  Improvement: {improvement:.1f}×")

# Step 4: 1000 requests
n_total = 1000
total_tokens = n_total * mean_output  # total useful tokens

# Static: process in batches of 8, each batch takes max_output / (gpu_throughput / batch_size) time
n_batches = int(np.ceil(n_total / batch_size))
time_per_batch = max_output * batch_size / gpu_throughput  # seconds
time_static = n_batches * time_per_batch

# Continuous: all tokens produced at full throughput
time_continuous = total_tokens / effective_continuous

print(f"\nWall-clock time for {n_total} requests:")
print(f"  Static:     {n_batches} batches × {time_per_batch:.1f}s = {time_static:.0f}s ({time_static/60:.1f} min)")
print(f"  Continuous: {total_tokens:,} tokens / {effective_continuous:.0f} tok/s = {time_continuous:.0f}s ({time_continuous/60:.1f} min)")
print(f"  Speedup:    {time_static/time_continuous:.1f}×")

---
## Exercise 8: Prefill-Decode Disaggregation

**Setup:**
- Prefill throughput: 1000 tokens/sec per GPU
- Decode throughput: 50 tokens/sec per GPU (batch=1)
- Workload: 100-token prompts + 500-token outputs

In a disaggregated architecture, prefill and decode run on separate GPU pools.
The key insight is matching throughput: prefill must generate KV caches fast
enough to keep decode GPUs busy.

**Tasks:**
1. Compute the time per request for prefill (process 100 input tokens).
2. Compute the time per request for decode (generate 500 output tokens).
3. If you have G total GPUs, compute the optimal ratio of prefill to decode GPUs.
4. For G=100 GPUs, compute the system throughput (requests/sec).

In [ ]:
# === Exercise 8: Scaffold ===

prefill_throughput = 1000  # tokens/sec per GPU
decode_throughput = 50     # tokens/sec per GPU
prompt_len = 100
output_len = 500

# TODO 1: Time per request for prefill
t_prefill = None  # prompt_len / prefill_throughput

# TODO 2: Time per request for decode
t_decode = None  # output_len / decode_throughput

# TODO 3: Optimal GPU ratio
#   Match throughput: prefill_gpus / t_prefill = decode_gpus / t_decode
#   Ratio: prefill_gpus / decode_gpus = t_prefill / t_decode
ratio = None  # t_prefill / t_decode

# For G total GPUs: prefill_gpus + decode_gpus = G
#   prefill_gpus = G * ratio / (1 + ratio)
#   decode_gpus  = G / (1 + ratio)
G = 100
n_prefill = None  # fill in
n_decode = None   # fill in

# TODO 4: System throughput (requests/sec)
#   Bottleneck = min(prefill capacity, decode capacity)
prefill_capacity = None  # n_prefill / t_prefill
decode_capacity = None   # n_decode / t_decode
system_throughput = None  # min of the two

In [ ]:
# === Exercise 8: Solution ===
np.random.seed(42)

print("=== Exercise 8: Prefill-Decode Disaggregation ===\n")

prefill_throughput = 1000  # tokens/sec per GPU
decode_throughput = 50     # tokens/sec per GPU
prompt_len = 100
output_len = 500

# Step 1: Time per request on each stage
t_prefill = prompt_len / prefill_throughput
t_decode = output_len / decode_throughput
print(f"Time per request:")
print(f"  Prefill: {prompt_len} tokens / {prefill_throughput} tok/s = {t_prefill:.3f}s")
print(f"  Decode:  {output_len} tokens / {decode_throughput} tok/s = {t_decode:.1f}s")
print(f"  Decode is {t_decode/t_prefill:.0f}× slower per request")

# Step 2: Optimal ratio
ratio = t_prefill / t_decode
print(f"\nOptimal GPU ratio:")
print(f"  prefill/decode = t_prefill/t_decode = {t_prefill:.3f}/{t_decode:.1f} = {ratio:.4f}")
print(f"  ≈ 1 prefill GPU per {1/ratio:.0f} decode GPUs")

# Step 3: For G=100
G = 100
n_prefill = G * ratio / (1 + ratio)
n_decode = G / (1 + ratio)
print(f"\nFor G={G} GPUs:")
print(f"  Prefill GPUs: {G} × {ratio:.4f} / {1+ratio:.4f} = {n_prefill:.1f} → {int(np.round(n_prefill))}")
print(f"  Decode GPUs:  {G} / {1+ratio:.4f} = {n_decode:.1f} → {int(np.round(n_decode))}")

n_pf = int(np.round(n_prefill))
n_dc = G - n_pf
print(f"  Allocation: {n_pf} prefill + {n_dc} decode = {G} total")

# Step 4: System throughput
prefill_cap = n_pf / t_prefill  # requests/sec (prefill capacity)
decode_cap = n_dc / t_decode    # requests/sec (decode capacity)
system_thr = min(prefill_cap, decode_cap)

print(f"\nSystem throughput:")
print(f"  Prefill capacity: {n_pf} GPUs / {t_prefill:.3f}s = {prefill_cap:.1f} req/s")
print(f"  Decode capacity:  {n_dc} GPUs / {t_decode:.1f}s = {decode_cap:.1f} req/s")
print(f"  System throughput: {system_thr:.1f} req/s (limited by {'prefill' if prefill_cap < decode_cap else 'decode'})")

# Comparison: non-disaggregated
t_total = t_prefill + t_decode
non_disagg = G / t_total  # all GPUs handle full requests
print(f"\nComparison: non-disaggregated system:")
print(f"  {G} GPUs × 1 req / {t_total:.3f}s = {non_disagg:.1f} req/s")
print(f"  Disaggregation advantage: {system_thr/non_disagg:.2f}× {'better' if system_thr > non_disagg else 'worse'}")
print(f"  (Disaggregation avoids interference between compute-bound prefill")
print(f"   and bandwidth-bound decode phases)")

print("\n" + "=" * 60)
print("ALL EXERCISES COMPLETE")
print("=" * 60)